# Étape 5 — FULL : gradient complet de Tweedie

**Un prompt dev, seed 0, luminance cible 0,7.** SD1.5, 512×512, DDIM 50, eta=0, CFG=7.5.
FULL calcule le gradient du surrogate de Tweedie, **pas le score cible exact**.
Le graphe traverse les deux branches du U-Net, puis le VAE et la récompense. Il est libéré à chaque pas.

1. Dans un runtime Colab **GPU T4 dédié**, importer `diffusion-reward-guidance.zip` à la première cellule.
2. Exécuter les deux premières cellules de code. Si une installation demande un redémarrage, le faire
   avant de reprendre au début. Sinon, poursuivre sans redémarrer.
3. Le notebook génère BASE, teste **un seul pas FULL**, puis seulement les contrôles et comparaisons complètes.
4. En cas d'OOM sur ce premier pas, il sauvegarde l'échec et tente une fois le checkpointing standard
   U-Net + VAE. Si cela échoue encore, exporter les diagnostics et reprendre le même code sur **H100 80 Go**.
5. Télécharger les résultats avec la dernière cellule, également utilisable après un échec.

Les anciens modules et notebooks sont conservés. Aucun résultat FULL GPU n'est prérempli.
Une récompense inférieure à COV ou BASE ne fait pas échouer les tests algorithmiques.

In [ ]:
from pathlib import Path
import os
import sys
import zipfile

candidates = [Path.cwd(), Path.cwd().parent, Path('/content/diffusion-reward-guidance')]
PROJECT_DIR = next((p for p in candidates if (p / 'guidance_full.py').is_file()), None)
if PROJECT_DIR is None:
    from google.colab import files
    print('Importer diffusion-reward-guidance.zip.')
    uploaded = files.upload()
    archive_name = next(name for name in uploaded if name.endswith('.zip'))
    destination = Path('/content').resolve()
    with zipfile.ZipFile(archive_name) as archive:
        for entry in archive.infolist():
            (destination / entry.filename).resolve().relative_to(destination)
        archive.extractall(destination)
    PROJECT_DIR = destination / 'diffusion-reward-guidance'
assert (PROJECT_DIR / 'guidance_full.py').is_file(), 'Archive FULL requise.'
os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
print('Projet :', PROJECT_DIR)

## 1. Versions figées

Même environnement que COV : aucune dépendance supplémentaire. Les installations sont journalisées.
ImageReward n'est pas chargé ; le pilote n'utilise que la luminance. Sur H100, ouvrir ce même notebook
depuis le dossier du projet avec un kernel GPU, puis utiliser les mêmes cellules. Le GPU réel est enregistré.

In [ ]:
import importlib.metadata
import subprocess

assert (3, 10) <= sys.version_info[:2] <= (3, 13), 'Python 3.10 à 3.13 requis.'
expected = dict(line.strip().split('==') for line in
                (PROJECT_DIR / 'requirements.txt').read_text(encoding='utf-8').splitlines()
                if line.strip() and not line.lstrip().startswith('#'))
torch_expected = {'torch': '2.8.0+cu126', 'torchvision': '0.23.0+cu126', 'torchaudio': '2.8.0+cu126'}
expected.update(torch_expected)
installed = {}
for package in expected:
    try:
        installed[package] = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        installed[package] = None
needs_install = any(installed[p] != v for p, v in expected.items())
RESTART_REQUIRED = globals().get('RESTART_REQUIRED', False) or needs_install

def run_visible(command, log_name):
    log_dir = PROJECT_DIR / 'installation_logs'
    log_dir.mkdir(exist_ok=True)
    with (log_dir / log_name).open('w', encoding='utf-8') as log:
        with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, encoding='utf-8', errors='replace', bufsize=1) as process:
            for line in process.stdout:
                print(line, end='', flush=True)
                log.write(line)
            return_code = process.wait()
    if return_code:
        raise RuntimeError(f'Échec {return_code} : voir les lignes ci-dessus et {log_dir / log_name}.')

if needs_install:
    if any(installed[p] != v for p, v in torch_expected.items()):
        run_visible([sys.executable, '-m', 'pip', 'install', '--progress-bar', 'off', '--only-binary=:all:',
                     'torch==2.8.0', 'torchvision==0.23.0', 'torchaudio==2.8.0',
                     '--index-url', 'https://download.pytorch.org/whl/cu126'], 'full-pytorch.log')
    run_visible([sys.executable, '-m', 'pip', 'install', '--progress-bar', 'off', '--only-binary=:all:',
                 '-r', 'requirements.txt'], 'full-dependencies.log')
print('STOP : redémarrer la session puis reprendre au début.' if RESTART_REQUIRED
      else 'Versions conformes : poursuivre, sans redémarrage.')

In [ ]:
assert not RESTART_REQUIRED, 'Redémarrer réellement la session avant les imports.'
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
import csv
import gc
import hashlib
import inspect
import json
import math
import platform
import shutil
import time
from datetime import datetime, timezone
import torch
from PIL import Image, ImageDraw
from IPython.display import display
from baseline import (MODEL_ID, MODEL_REVISION, configure_torch, load_pipeline,
                      encode_prompt_cfg, make_initial_noise, sample_ddim, measure_call)
from rewards import decode_latents_differentiable, luminance_map, luminance_reward
from guidance_cov import cov_noise_prediction, sample_cov
from guidance_full import full_noise_prediction, sample_full

configure_torch()
assert torch.cuda.is_available(), 'Sélectionner un GPU T4 dans Colab.'
print(torch.cuda.get_device_name(0), torch.__version__, torch.version.cuda)
RUN_DIR = PROJECT_DIR / 'runs' / datetime.now(timezone.utc).strftime('full_%Y%m%d_%H%M%S_%f_UTC')
RUN_DIR.mkdir(parents=True, exist_ok=False)
results = []
algorithm_checks = {'gpu_status': 'pending', 'trajectories': {}}

def save_json(name, value):
    (RUN_DIR / name).write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False), encoding='utf-8')

run_visible([sys.executable, '-m', 'unittest', '-v', 'test_baseline', 'test_rewards', 'test_cov', 'test_full'], 'full-tests.log')
shutil.copy2(PROJECT_DIR / 'installation_logs' / 'full-tests.log', RUN_DIR / 'cpu_tests.txt')
print('Dossier résultats :', RUN_DIR)
probe_reports = []
FULL_READY = False

## 2. Modèle et entrées partagées

Comme COV, les états DDIM et le U-Net restent fp16. FULL crée une feuille $z_t$ fp32, puis la convertit
vers le U-Net **sans detach**. Les deux branches CFG conservent les opérations de BASE.
Le calcul de $\mu_t$, le VAE, le gradient retourné et la correction sont fp32.

**Le backward interne du U-Net reste fp16.** La feuille fp32 ne supprime pas ses arrondis ou risques de
sous-flux. Les différences finies de référence sont donc testées sur un petit réseau réellement fp32
dans `test_full.py`, pas interprétées naïvement à travers les arrondis fp16 du vrai SD1.5.

Le VAE utilise les mêmes poids de variante fp16 convertis en fp32 que le pilote COV. Tous les paramètres
restent gelés en `eval()`. Le checkpointing est désactivé au départ ; un changement sera documenté.

In [ ]:
SEED, NUM_STEPS, CFG, TARGET, TAU2 = 0, 50, 7.5, 0.7, 1.0
dev = (PROJECT_DIR / 'data/prompts_50_dev.txt').read_text(encoding='utf-8-sig').splitlines()
heldout = (PROJECT_DIR / 'data/prompts_50_heldout.txt').read_text(encoding='utf-8-sig').splitlines()
all_prompts = (PROJECT_DIR / 'data/generation_prompts.txt').read_text(encoding='utf-8-sig').splitlines()
assert not set(dev) & set(heldout)
PROMPT = dev[0]
assert PROMPT in all_prompts
pipe = load_pipeline()
pipe.vae.to(dtype=torch.float32).eval().requires_grad_(False)
assert not pipe.vae.is_gradient_checkpointing
prompt_embeds, negative_prompt_embeds = encode_prompt_cfg(pipe, PROMPT)
initial_noise = make_initial_noise(pipe, SEED)
assert initial_noise.shape == (1, 4, 64, 64)
assert torch.equal(initial_noise, make_initial_noise(pipe, SEED))

def reward_fn(images):
    return luminance_reward(images, target=TARGET)

torch.save({'initial_noise': initial_noise.cpu(), 'prompt_embeds': prompt_embeds.cpu(),
            'negative_prompt_embeds': negative_prompt_embeds.cpu()}, RUN_DIR / 'inputs.pt')
with torch.no_grad():
    warm_latents, _ = sample_ddim(pipe, initial_noise, prompt_embeds, negative_prompt_embeds, num_steps=2)
    warm_image = decode_latents_differentiable(pipe.vae, warm_latents.float())
del warm_latents, warm_image
gc.collect()
torch.cuda.empty_cache()
pipe.scheduler.set_timesteps(NUM_STEPS, device=initial_noise.device)
source_files = ['baseline.py', 'rewards.py', 'guidance_cov.py', 'test_baseline.py',
                'test_rewards.py', 'test_cov.py', 'guidance_full.py', 'test_full.py',
                'requirements.txt', 'notebooks/04_full_t4.ipynb']
snapshot = RUN_DIR / 'source'
for name in source_files:
    (snapshot / name).parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(PROJECT_DIR / name, snapshot / name)
environment = {
    'model_id': MODEL_ID, 'model_revision': MODEL_REVISION, 'prompt': PROMPT, 'seed': SEED,
    'resolution': [512, 512], 'num_steps': NUM_STEPS, 'eta': 0, 'cfg': CFG, 'target': TARGET, 'tau2': TAU2,
    'unet_dtype': str(pipe.unet.dtype), 'vae_dtype': str(pipe.vae.dtype), 'state_dtype': str(initial_noise.dtype),
    'correction_dtype': 'torch.float32; cast to state dtype before DDIM',
    'initial_gradient_checkpointing': False, 'offload': False, 'reward': 'luminance',
    'unet_backward_dtype': 'torch.float16', 'final_gpu_target': 'H100 80 GB',
    'scheduler': dict(pipe.scheduler.config), 'timesteps': pipe.scheduler.timesteps.cpu().tolist(),
    'gpu': torch.cuda.get_device_name(0), 'vram_mib': torch.cuda.get_device_properties(0).total_memory / 2**20,
    'python': platform.python_version(), 'torch': torch.__version__, 'cuda': torch.version.cuda,
    'deterministic_algorithms': torch.are_deterministic_algorithms_enabled(),
    'versions': {package: importlib.metadata.version(package) for package in expected},
    'source_sha256': {name: hashlib.sha256((PROJECT_DIR / name).read_bytes()).hexdigest() for name in source_files},
    'prompt_sha256': {name: hashlib.sha256((PROJECT_DIR / 'data' / name).read_bytes()).hexdigest()
                      for name in ['generation_prompts.txt', 'prompts_50_dev.txt', 'prompts_50_heldout.txt']},
    'cost_scope': 'sampler + diagnostics + CPU trace + final fp32 VAE decode; excludes disk/model load/text encoding/warmup',
}
save_json('environment.json', environment)
(RUN_DIR / 'pip_freeze.txt').write_text(subprocess.check_output(
    [sys.executable, '-m', 'pip', 'freeze'], text=True), encoding='utf-8')
print(PROMPT)
print(inspect.getsource(full_noise_prediction))

## 3. BASE et mesure commune

Le wrapper ci-dessous appelle les modules existants et sauvegarde chaque expérience séparément.
Le temps inclut sampler, diagnostics, traces CPU et décodage final ; il exclut chargement, texte,
chauffe et écritures disque. La préparation FULL/calibration sera chiffrée séparément.
Le VAE fp32 décode toutes les images finales pour comparer le même score.

In [ ]:
def run_generation(label, method, beta=0.0, selected_reward=reward_fn):
    if method == 'FULL' and beta != 0 and not FULL_READY:
        raise RuntimeError('Valider le pas FULL isolé avant une trajectoire différentiable.')
    if (RUN_DIR / f'{label}_steps.json').exists():
        raise FileExistsError(f'{label} existe déjà ; utiliser un nouveau RUN_DIR pour répéter.')
    step_reports = []
    checkpoint_flags = {'unet_checkpointing': pipe.unet.is_gradient_checkpointing,
                        'vae_checkpointing': pipe.vae.is_gradient_checkpointing}

    def generate_and_decode():
        arguments = dict(num_steps=NUM_STEPS, guidance_scale=CFG, return_trace=True)
        if method == 'BASE':
            final_latents, trace = sample_ddim(pipe, initial_noise, prompt_embeds, negative_prompt_embeds, **arguments)
        elif method == 'COV':
            final_latents, trace, _ = sample_cov(pipe, initial_noise, prompt_embeds, negative_prompt_embeds,
                selected_reward, beta=beta, tau2=TAU2, diagnostics=step_reports, **arguments)
        elif method == 'FULL':
            final_latents, trace, _ = sample_full(pipe, initial_noise, prompt_embeds, negative_prompt_embeds,
                selected_reward, beta=beta, diagnostics=step_reports, **arguments)
        else:
            raise ValueError(method)
        with torch.no_grad():
            image = decode_latents_differentiable(pipe.vae, final_latents.float()).cpu()
        return final_latents.cpu(), trace, image

    started = time.perf_counter()
    try:
        (latents, trace, image), cost = measure_call(generate_and_decode)
        if not torch.isfinite(latents).all() or not torch.isfinite(image).all():
            raise FloatingPointError('Latents ou image finale non finis.')
        mean_luminance = luminance_map(image).mean().item()
        reward = reward_fn(image).item()
    except Exception as error:
        save_json(f'{label}_failure.json', {'method': method, 'beta': beta, 'type': type(error).__name__,
                  'message': str(error), 'elapsed_seconds': time.perf_counter()-started, **checkpoint_flags})
        save_json(f'{label}_steps.json', step_reports)
        if isinstance(error, torch.cuda.OutOfMemoryError):
            (RUN_DIR / f'{label}_OOM.txt').write_text(torch.cuda.memory_summary(), encoding='utf-8')
        algorithm_checks['gpu_status'] = 'failed'
        save_json('algorithm_checks.json', algorithm_checks)
        raise

    if method == 'BASE':
        steps_valid = True
    else:
        graph_key = 'unet_graph_present' if method == 'FULL' else 'unet_graph_absent'
        steps_valid = len(step_reports) == NUM_STEPS and all(
            s['finite'] and s['state_detached'] and (beta == 0 or s[graph_key]) for s in step_reports)
    connected = beta == 0 or all(s['reward_connected'] for s in step_reports)
    algorithm_checks['trajectories'][label] = dict(completed_and_finite=True,
                                                  steps_valid=steps_valid, reward_connected=connected)
    if not (steps_valid and connected):
        algorithm_checks['gpu_status'] = 'failed'
    save_json('algorithm_checks.json', algorithm_checks)
    row = dict(label=label, method=method, beta=beta, reward=reward,
               delta_reward_vs_base=reward-(results[0]['reward'] if results else reward),
               mean_luminance=mean_luminance, absolute_error=abs(mean_luminance-TARGET),
               strong_correction_steps=sum(s['strong_correction'] for s in step_reports),
               max_relative_correction=max((s['relative_noise_correction'] for s in step_reports), default=0),
               **checkpoint_flags, **cost)
    data = dict(latents=latents, trace=trace, image=image)
    torch.save(data, RUN_DIR / f'{label}_tensors.pt')
    pixels = (image[0].permute(1, 2, 0).numpy()*255).round().clip(0,255).astype('uint8')
    Image.fromarray(pixels).save(RUN_DIR / f'{label}.png')
    save_json(f'{label}_steps.json', step_reports)
    if step_reports:
        with (RUN_DIR / f'{label}_steps.csv').open('w', newline='', encoding='utf-8') as stream:
            writer = csv.DictWriter(stream, fieldnames=list(step_reports[0]))
            writer.writeheader()
            writer.writerows(step_reports)
    results.append(row)
    save_json('results.json', results)
    with (RUN_DIR / 'results.csv').open('w', newline='', encoding='utf-8') as stream:
        writer = csv.DictWriter(stream, fieldnames=list(row))
        writer.writeheader()
        writer.writerows(results)
    print(json.dumps(row, indent=2))
    assert steps_valid and connected, 'Diagnostic de trajectoire invalide : lire les rapports sauvegardés.'
    return data, step_reports

base, _ = run_generation('BASE', 'BASE')
display(Image.open(RUN_DIR / 'BASE.png'))

## 4. Un seul pas FULL avant toute trajectoire guidée

Mesurer d'abord au premier état BASE, $t=981$, avec $\beta=1$. Le gradient retourné n'est pas multiplié
par beta : sa norme est directement inspectable. Un hook sur la sortie du U-Net vérifie que le backward
atteint **ses deux branches**, sans calculer de gradients de poids.
Les normes de leurs gradients sont enregistrées, y compris lorsqu'elles sont nulles : un point
stationnaire ne prouve pas un bug. Une calibration entièrement nulle empêchera de définir les beta.

Le ratio utilisé pour la future calibration est calculé en fp32 **avant conversion du bruit corrigé**.
À beta=1, l'arrondi fp16 peut atténuer fortement la correction ; cela n'annule pas le gradient mesuré.

Un OOM déclenche un seul nouvel essai avec `enable_gradient_checkpointing()` sur le U-Net et le VAE,
toujours gelés en `eval()`. Cela recalcule des activations pendant le backward, sans remplacer FULL.
Les deux essais sont archivés. Pas de diminution silencieuse de résolution, de pas ou de méthode.

In [ ]:
def probe_full(index, label):
    # Un pas, scalaires et gradient CPU seulement ; aucun traceback conservé.
    if any(p['label'] == label for p in probe_reports) or (RUN_DIR / f'{label}_gradient.pt').exists():
        raise FileExistsError(f'{label} existe déjà ; utiliser un nouveau RUN_DIR pour répéter.')
    backward = {'seen': False, 'branch_l2': None}

    def observe_unet(_module, _inputs, outputs):
        if outputs[0].requires_grad:
            def record_gradient(gradient):
                backward['seen'] = True
                backward['branch_l2'] = [g.float().norm().item() for g in gradient.detach().chunk(2)]
            outputs[0].register_hook(record_gradient)

    pipe.scheduler.set_timesteps(NUM_STEPS, device=initial_noise.device)
    timestep = pipe.scheduler.timesteps[index]
    state = (initial_noise * pipe.scheduler.init_noise_sigma if index == 0
             else base['trace'][index-1].to(initial_noise.device))
    flags = {'unet_checkpointing': pipe.unet.is_gradient_checkpointing,
             'vae_checkpointing': pipe.vae.is_gradient_checkpointing}
    handle = pipe.unet.register_forward_hook(observe_unet)
    started = time.perf_counter()
    try:
        (noise_guided, gradient, report), cost = measure_call(
            full_noise_prediction, pipe, state, timestep, prompt_embeds, negative_prompt_embeds,
            reward_fn, beta=1.0, guidance_scale=CFG)
        assert gradient is not None and gradient.shape == state.shape and torch.isfinite(gradient).all()
        assert report['unet_graph_present'] and report['reward_connected'] and backward['seen']
        assert len(backward['branch_l2']) == 2 and all(math.isfinite(n) for n in backward['branch_l2'])
        backward['branch_nonzero'] = [n > 0 for n in backward['branch_l2']]
        report['gradient_nonzero'] = report['gradient_l2'] > 0
        assert all(p.grad is None for model in (pipe.unet, pipe.vae) for p in model.parameters())
        torch.save(gradient.cpu(), RUN_DIR / f'{label}_gradient.pt')
        report.update(label=label, method='FULL', step_index=index, status='passed', cost=cost,
                      unet_backward=backward, **flags)
        del noise_guided, gradient
    except Exception as error:
        report = dict(label=label, method='FULL', step_index=index, timestep=int(timestep),
                      status='oom' if isinstance(error, torch.cuda.OutOfMemoryError) else 'failed',
                      error_type=type(error).__name__, message=str(error),
                      elapsed_seconds=time.perf_counter()-started, **flags)
        if torch.cuda.is_available():
            report['peak_allocated_mib'] = torch.cuda.max_memory_allocated()/2**20
            report['peak_reserved_mib'] = torch.cuda.max_memory_reserved()/2**20
        if report['status'] == 'oom':
            (RUN_DIR / f'{label}_OOM.txt').write_text(torch.cuda.memory_summary(), encoding='utf-8')
    finally:
        handle.remove()
    probe_reports.append(report)
    save_json('probes.json', probe_reports)
    print(json.dumps(report, indent=2))
    return report

FULL_READY = False
algorithm_checks.update(gpu_status='single_step_pending', full_single_step_passed=False,
                        identities_passed=False, calibration_passed=False, all_trajectories_valid=False)
save_json('algorithm_checks.json', algorithm_checks)
first_probe = probe_full(0, 'FULL_t981_plain')
if first_probe['status'] == 'oom':
    print('OOM archivé. Nouvel essai explicite avec checkpointing U-Net + VAE ; FULL reste inchangé.')
    gc.collect()
    torch.cuda.empty_cache()
    pipe.unet.enable_gradient_checkpointing()
    pipe.vae.enable_gradient_checkpointing()
    first_probe = probe_full(0, 'FULL_t981_checkpoint')
FULL_READY = first_probe['status'] == 'passed'
algorithm_checks['full_single_step_passed'] = FULL_READY
algorithm_checks['memory_profile'] = {'unet_checkpointing': pipe.unet.is_gradient_checkpointing,
                                     'vae_checkpointing': pipe.vae.is_gradient_checkpointing}
algorithm_checks['gpu_status'] = ('single_step_passed_trajectory_pending' if FULL_READY else
                                 'oom_prepare_H100' if first_probe['status'] == 'oom' else 'failed')
save_json('algorithm_checks.json', algorithm_checks)
if not FULL_READY:
    raise RuntimeError('FULL non validé sur ce GPU. Exporter avec la dernière cellule ; analyser ou reprendre sur H100 80 Go.')
print('Un pas FULL validé. Les contrôles sur 50 pas peuvent commencer.')

## 5. Identités contrôlées : même BASE quand la correction est nulle

Comparer les 50 états, latents et images finaux, sans tolérance élargie. Beta=0 ne construit aucun
graphe de récompense. La constante reliée au graphe vérifie un vrai backward nul.
Si le checkpointing a été nécessaire, il reste actif pour les comparaisons COV/FULL suivantes ;
chaque ligne de résultat conserve les deux indicateurs. Ne pas mélanger ces coûts avec le pilote COV antérieur.

In [ ]:
algorithm_checks.update(gpu_status='identities_pending', identities_passed=False,
                        calibration_passed=False, all_trajectories_valid=False)
save_json('algorithm_checks.json', algorithm_checks)
assert FULL_READY, 'Valider le pas FULL isolé avant les identités.'
for label, method, beta, selected_reward in [
    ('FULL_beta0', 'FULL', 0.0, reward_fn),
    ('COV_beta0', 'COV', 0.0, reward_fn),
    ('FULL_constant', 'FULL', 1.0, lambda images: 1 + 0*images.mean((1,2,3))),
]:
    data, steps = run_generation(label, method, beta, selected_reward)
    exact_states = (len(data['trace']) == len(base['trace']) == NUM_STEPS
                    and all(torch.equal(a,b) for a,b in zip(base['trace'], data['trace'])))
    comparison = {'states_exact': exact_states, 'latents_exact': torch.equal(data['latents'], base['latents']),
                  'image_exact': torch.equal(data['image'], base['image']),
                  'per_step_max_abs': [(a.float()-b.float()).abs().max().item()
                                      for a,b in zip(base['trace'],data['trace'])]}
    if label == 'FULL_constant':
        comparison['gradient_zero'] = all(s['gradient_l2'] == 0 for s in steps)
    algorithm_checks[label] = comparison
    good = comparison['states_exact'] and comparison['latents_exact'] and comparison['image_exact'] and comparison.get('gradient_zero', True)
    if not good:
        algorithm_checks['gpu_status'] = 'failed'
    save_json('algorithm_checks.json', algorithm_checks)
    assert good, f'Identité {label} échouée : lire les différences sauvegardées.'
    del data, steps
algorithm_checks['identities_passed'] = True
save_json('algorithm_checks.json', algorithm_checks)

## 6. Mesurer FULL et COV aux mêmes états BASE

Trois états d'entrée : indices 0, 24, 49. Le premier gradient FULL est réutilisé ; les deux autres sont
mesurés avant les comparaisons finales. Pour chaque méthode, estimer
$q_t=\|\sigma_t g_t(\beta=1)\|/\|\epsilon_{CFG}\|$, puis $q_{max}$ sur ces sondes.

Deux comparaisons sont fixées **avant** de voir les images guidées :

- **Même beta** : $\beta_{commun}=0.01/\max(q_{max}^{FULL},q_{max}^{COV})$.
- **Corrections calibrées à 1 %** : $\beta_{méthode}=0.01/q_{max}^{méthode}$.

Cette règle évite de reprendre les grands beta de COV sans mesurer FULL. « Corrections comparables »
signifie seulement le même maximum sur les trois sondes BASE ; les trajectoires peuvent ensuite diverger.
Aucun gradient n'est normalisé ou tronqué pendant la génération. Une correction >10 % est signalée.

In [ ]:
@torch.no_grad()
def probe_cov(index):
    timestep = pipe.scheduler.timesteps[index]
    state = (initial_noise*pipe.scheduler.init_noise_sigma if index == 0
             else base['trace'][index-1].to(initial_noise.device))
    embeddings = torch.cat([negative_prompt_embeds, prompt_embeds])
    latent_input = pipe.scheduler.scale_model_input(torch.cat([state,state]), timestep)
    noise = pipe.unet(latent_input, timestep, encoder_hidden_states=embeddings, return_dict=False)[0]
    noise_u, noise_c = noise.chunk(2)
    noise_cfg = noise_u + CFG*(noise_c-noise_u)
    _, report = cov_noise_prediction(pipe.vae, state, noise_cfg, timestep, pipe.scheduler, reward_fn, beta=1.0, tau2=TAU2)
    return report

algorithm_checks.update(gpu_status='calibration_pending', calibration_passed=False, all_trajectories_valid=False)
save_json('algorithm_checks.json', algorithm_checks)
assert FULL_READY and algorithm_checks.get('identities_passed'), 'Valider le pas isolé et les identités avant la calibration.'
PROBE_INDICES = [0, NUM_STEPS//2-1, NUM_STEPS-1]
full_probes = [first_probe]
cov_probes = []
for index in PROBE_INDICES:
    if index != 0:
        full_probe = probe_full(index, f'FULL_t{int(pipe.scheduler.timesteps[index])}')
        if full_probe['status'] != 'passed':
            algorithm_checks['gpu_status'] = 'failed_during_calibration'
            save_json('algorithm_checks.json', algorithm_checks)
            raise RuntimeError('Échec FULL sur une autre sonde. Exporter ; aucune génération positive lancée.')
        full_probes.append(full_probe)
    try:
        cov_probe, cost = measure_call(probe_cov, index)
    except Exception as error:
        save_json('cov_calibration_failure.json', {'step_index': index, 'type': type(error).__name__, 'message': str(error)})
        algorithm_checks['gpu_status'] = 'failed_during_calibration'
        save_json('algorithm_checks.json', algorithm_checks)
        if isinstance(error, torch.cuda.OutOfMemoryError):
            (RUN_DIR / 'cov_calibration_OOM.txt').write_text(torch.cuda.memory_summary(), encoding='utf-8')
        raise
    cov_probe.update(method='COV', step_index=index, status='passed', cost=cost)
    cov_probes.append(cov_probe)
    save_json('calibration.json', {'status': 'measuring', 'full_probes': full_probes, 'cov_probes': cov_probes})

q_full = max(p['relative_noise_correction_fp32'] for p in full_probes)
q_cov = max(p['relative_noise_correction_fp32'] for p in cov_probes)
if not all(math.isfinite(q) and q > 0 for q in (q_full, q_cov)):
    algorithm_checks['gpu_status'] = 'calibration_unavailable'
    save_json('algorithm_checks.json', algorithm_checks)
    raise RuntimeError('Sondes sauvegardées mais grille non définie : gradient nul/inexploitable à examiner, pas preuve de bug.')
TARGET_RATIO = 0.01
BETA_SHARED = TARGET_RATIO/max(q_full,q_cov)
BETA_FULL_MATCHED = TARGET_RATIO/q_full
BETA_COV_MATCHED = TARGET_RATIO/q_cov
calibration = {'status': 'grid_fixed_before_generations', 'target_ratio': TARGET_RATIO,
               'q_full': q_full, 'q_cov': q_cov, 'beta_shared': BETA_SHARED,
               'beta_full_matched': BETA_FULL_MATCHED, 'beta_cov_matched': BETA_COV_MATCHED,
               'full_probes': full_probes, 'cov_probes': cov_probes,
               'successful_probe_seconds': sum(p['cost']['seconds'] for p in full_probes+cov_probes),
               'failed_attempts': [p for p in probe_reports if p['status'] != 'passed']}
save_json('calibration.json', calibration)
algorithm_checks['calibration_passed'] = True
save_json('algorithm_checks.json', algorithm_checks)
print(json.dumps({k:v for k,v in calibration.items() if k not in ('full_probes','cov_probes','failed_attempts')}, indent=2))

## 7. Comparaisons complètes

Chaque méthode utilise le même bruit initial, prompt, seed, scheduler, résolution et récompense.
Si une paire méthode/beta est identique entre les deux comparaisons, réutiliser son résultat et enregistrer
cette correspondance. Aucun résultat n'est choisi selon son score. Le coût d'un résultat réutilisé est
celui de la génération mesurée, sans prétendre avoir réalisé une seconde répétition.

Un OOM ou NaN pendant une trajectoire est sauvegardé puis arrête cette cellule. Aucun remplacement
de FULL par COV ni adaptation silencieuse de beta en cours de génération.

In [ ]:
algorithm_checks.update(gpu_status='comparisons_pending', all_trajectories_valid=False)
save_json('algorithm_checks.json', algorithm_checks)
assert FULL_READY and algorithm_checks.get('calibration_passed') and algorithm_checks.get('identities_passed')
comparison_cases = [
    ('COV_shared', 'COV', BETA_SHARED), ('FULL_shared', 'FULL', BETA_SHARED),
    ('COV_matched', 'COV', BETA_COV_MATCHED), ('FULL_matched', 'FULL', BETA_FULL_MATCHED),
]
aliases = {}
completed = {}
for label, method, beta in comparison_cases:
    key = (method, beta)
    if key in completed:
        aliases[label] = completed[key]
    else:
        generated, steps = run_generation(label, method, beta)
        completed[key] = label
        aliases[label] = label
        del generated, steps
    save_json('comparison_aliases.json', aliases)

required_labels = ['BASE','FULL_beta0','COV_beta0','FULL_constant'] + list(set(aliases.values()))
algorithm_checks['all_trajectories_valid'] = all(
    label in algorithm_checks['trajectories'] and all(algorithm_checks['trajectories'][label].values())
    for label in required_labels)
algorithm_checks['models_frozen'] = all(not p.requires_grad for model in (pipe.unet,pipe.vae) for p in model.parameters())
algorithm_checks['parameter_grads_none'] = all(p.grad is None for model in (pipe.unet,pipe.vae) for p in model.parameters())
passed = (algorithm_checks.get('full_single_step_passed',False) and algorithm_checks.get('identities_passed',False)
          and algorithm_checks.get('calibration_passed',False) and len(aliases) == 4
          and algorithm_checks['all_trajectories_valid'] and algorithm_checks['models_frozen']
          and algorithm_checks['parameter_grads_none'])
algorithm_checks['gpu_status'] = 'passed_for_this_pilot' if passed else 'failed'
algorithm_checks['scope'] = 'algorithmic checks and complete trajectories; reward improvement is not a success condition'
save_json('algorithm_checks.json', algorithm_checks)
assert passed, 'Validation incomplète ou échouée ; consulter algorithm_checks.json.'
print('Contrôles algorithmiques réussis pour ce pilote ; comparer maintenant les scores et les coûts.')

## 8. BASE, COV, FULL côte à côte

La première figure compare le **même beta** ; la seconde compare des **corrections initialement calibrées**.
Un résultat FULL inférieur à COV est une observation expérimentale recevable. FULL calcule exactement
le gradient du surrogate à la précision numérique utilisée, mais ce surrogate et le modèle appris
n'intègrent pas exactement la récompense future. Une hausse du score ne prouve pas davantage la qualité générale.

In [ ]:
rows_by_label = {row['label']: row for row in results}
for comparison in ('shared','matched'):
    labels = ['BASE',aliases[f'COV_{comparison}'],aliases[f'FULL_{comparison}']]
    panel = Image.new('RGB', (1536,572), 'white')
    draw = ImageDraw.Draw(panel)
    for column,label in enumerate(labels):
        row = rows_by_label[label]
        panel.paste(Image.open(RUN_DIR / f'{label}.png'), (512*column,60))
        draw.text((512*column+8,6), f"{row['method']} | beta={row['beta']:.6g} | R={row['reward']:.6f}", fill='black')
        draw.text((512*column+8,24), f"L={row['mean_luminance']:.4f} | {row['seconds']:.1f}s | {row['peak_allocated_mib']/1024:.2f} GiB", fill='black')
        draw.text((512*column+8,42), f"delta R={row['delta_reward_vs_base']:+.6f}", fill='black')
    panel.save(RUN_DIR / f'compare_{comparison}.png')
    print('Même beta' if comparison == 'shared' else 'Corrections calibrées à 1 % sur trois états BASE')
    display(panel)
print('Préparation, sondes réussies (s) :', calibration['successful_probe_seconds'])
print('Échecs éventuels du premier essai :', len(calibration['failed_attempts']))

## 9. Export — à lancer aussi après un échec

L'archive contient les sources, entrées partagées, gradients des sondes FULL, traces, PNG, scores,
diagnostics, choix des beta et profils mémoire. Aucun token n'est exporté. Les OOM restent des échecs
mesurés, même si une nouvelle configuration permet ensuite de poursuivre.

Sur H100 80 Go, reprendre les mêmes fichiers et refaire les contrôles localement au GPU ; l'identité
numérique entre T4 et H100 n'est pas supposée. Le notebook peut être exécuté dans Jupyter sans Colab :
seuls l'import interactif du ZIP et son téléchargement final utilisent `google.colab`.

In [ ]:
save_json('algorithm_checks.json', algorithm_checks)
save_json('results.json', results)
archive_path = shutil.make_archive(str(RUN_DIR), 'zip', root_dir=RUN_DIR.parent, base_dir=RUN_DIR.name)
print('Résultats :', archive_path)
try:
    from google.colab import files
except ImportError:
    print('Northflank/local : récupérer cette archive depuis le système de fichiers.')
else:
    files.download(archive_path)